In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
products = pd.read_csv("../ecommerce_dataset/products.csv")

order_items = pd.read_csv("../ecommerce_dataset/order_items.csv")

In [3]:
recommendation_df = order_items.merge(
    products,
    on="product_id",
    how="left"
)

In [4]:
recommendation_df.head()

,order_item_id,order_id,product_id,user_id,quantity,item_price,item_total,product_name,category,brand,price,rating
0,I00000001,O00000001,P001758,U009310,2,8.07,16.14,Everest Whole,Pet Supplies,Everest,8.07,2.69
1,I00000002,O00000001,P001119,U009310,1,74.08,74.08,Nimbus Minute,Beauty,Nimbus,74.08,3.62
2,I00000003,O00000001,P001794,U009310,1,576.97,576.97,Willow Treatment,Automotive,Willow,576.97,3.89
3,I00000004,O00000001,P001038,U009310,1,22.47,22.47,Nimbus Deal,Books,Nimbus,22.47,4.85
4,I00000005,O00000002,P000859,U003247,1,422.22,422.22,Pulse Decide,Electronics,Pulse,422.22,3.51


In [13]:
product_user_matrix = user_product_matrix.T

In [14]:
user_product_matrix.shape
user_product_matrix.head()


product_id,P000001,P000002,P000003,P000004,P000005,P000006,P000007,P000008,P000009,P000010,...,P001991,P001992,P001993,P001994,P001995,P001996,P001997,P001998,P001999,P002000
user_id,,,,,,,,,,,,,,,,,,,,,
U000001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
U000002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
U000003,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
U000004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
U000005,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
product_similarity = cosine_similarity(product_user_matrix)
product_similarity.shape

(2000, 2000)

In [16]:
product_similarity_df = pd.DataFrame(
    product_similarity,
    index=product_user_matrix.index,
    columns=product_user_matrix.index
)

In [17]:
product_user_matrix.index[:10]

Index(['P000001', 'P000002', 'P000003', 'P000004', 'P000005', 'P000006',
       'P000007', 'P000008', 'P000009', 'P000010'],
      dtype='object', name='product_id')

In [22]:
product_similarity_df.loc['P000008'].sort_values(ascending=False).head(10)

product_id
P000008    1.000000
P001945    0.200545
P000912    0.137361
P001347    0.133697
P000191    0.130312
P001238    0.120217
P001119    0.106399
P000939    0.103835
P001532    0.099218
P000077    0.092726
Name: P000008, dtype: float64

In [23]:
def recommend_products(product_id, top_n=5):

    similar_products = product_similarity_df[product_id].sort_values(
        ascending=False
    )

    similar_products = similar_products.drop(product_id)

    top_products = similar_products.head(top_n)

    return top_products

In [24]:
products["product_id"].head()

0    P000001
1    P000002
2    P000003
3    P000004
4    P000005
Name: product_id, dtype: object

In [26]:
recommend_products('P000002')

product_id
P000400    0.209405
P001309    0.141983
P001548    0.125218
P001037    0.125218
P000421    0.123276
Name: P000002, dtype: float64

In [58]:
def recommend_product_names(product_id, top_n=5):

    original_product = products[
        products["product_id"] == product_id
    ]

    original_category = original_product["category"].values[0]
    original_brand = original_product["brand"].values[0]

    recommendations = recommend_products(product_id, top_n=50)

    result = (
        recommendations
        .reset_index()
        .rename(columns={
            "index": "product_id",
            product_id: "similarity"
        })
    )

    result = result.merge(
        products,
        on="product_id",
        how="left"
    )

    # نفس الفئة
    result = result[
        result["category"] == original_category
    ]

    # نفس البراند
    result["brand_match"] = (
        result["brand"] == original_brand
    ).astype(int)

    # الترتيب
    result = result.sort_values(
        by=["brand_match", "similarity", "rating"],
        ascending=[False, False, False]
    )

    return result.head(top_n)


In [60]:
recommend_product_names('P000049')

,product_id,similarity,product_name,category,brand,price,rating,brand_match
4,P001452,0.111456,Harbor Significant,Home & Kitchen,Harbor,505.82,2.92,0
8,P000481,0.082584,Harbor Enter,Home & Kitchen,Harbor,299.45,3.68,0
26,P000567,0.050084,Orion Democrat,Home & Kitchen,Orion,190.38,2.90,0
45,P001621,0.039058,Harbor College,Home & Kitchen,Harbor,15.11,4.76,0
46,P000918,0.038500,Astra Space,Home & Kitchen,Astra,98.55,4.11,0


In [41]:
products[products["product_id"] =='P000049']

,product_id,product_name,category,brand,price,rating
48,P000049,NeoTech Help,Home & Kitchen,NeoTech,12.72,4.6


In [42]:
recommendation_df["product_id"].nunique()


2000

In [37]:
recommendation_df["user_id"].nunique() 

8635